## XBRL US API - FERC schedule by entity  

### Authenticate for access token
Click in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title Authenticate XBRL API {"display-mode":"form"}
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

def refresh(info):
    refresh_auth = {
                'client_id': info.client_id,
                'client_secret' : info.client_secret,
                'grant_type' : 'refresh_token',
                'platform' : 'ipynb',
                'refresh_token' : info.refresh_token
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json.get('access_token')
    info.refresh_token = refresh_json.get('refresh_token')
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info

tokenInfo = tokenInfoClass()

# Helper to prompt only if value is missing
def prompt_if_missing(value, prompt_text, secret=False):
    if value:
        return value
    if secret:
        return getpass.getpass(prompt=prompt_text)
    return input(prompt_text)

# Load credentials (if .json exists)
creds = {}
if os.path.exists('creds.json'):
    try:
        with open('creds.json', 'r') as f:
            creds = json.load(f)
        print("Loaded .json")
    except Exception as e:
        print("Warning: failed to read from .json:", e)

if creds:
    # Try nested prod object first
    selected = None
    if isinstance(creds.get('prod'), dict):
        selected = creds['prod']
    
    # Next, try prod-prefixed keys
    if not selected:
        selected = {}
        keys = ['email', 'password', 'client_id', 'client_secret']
        for k in keys:
            prefixed_key = 'prod' + k
            if creds.get(prefixed_key):
                selected[k] = creds.get(prefixed_key)
            # fall back to top-level key if prod variant not found
            elif creds.get(k):
                selected[k] = creds.get(k)
    
    # Verify we have all required keys
    if not all(selected.get(k) for k in ('email', 'password', 'client_id', 'client_secret')):
        # Fill in missing values from prompts
        selected = {
            'email': selected.get('email'),
            'password': selected.get('password'),
            'client_id': selected.get('client_id'),
            'client_secret': selected.get('client_secret')
        }
    
    # Assign values, prompting for any missing ones
    tokenInfo.email = prompt_if_missing(selected.get('email'), 'Enter your XBRL US Web account email: ')
    tokenInfo.password = prompt_if_missing(selected.get('password'), 'Password: ', secret=True)
    tokenInfo.client_id = prompt_if_missing(selected.get('client_id'), 'Client ID: ', secret=True)
    tokenInfo.client_secret = prompt_if_missing(selected.get('client_secret'), 'Secret: ', secret=True)

    print('Using credentials from .json as available.')
else:
    # No creds.json — prompt the user
    tokenInfo.email = input('Enter your XBRL US Web account email: ')
    tokenInfo.password = getpass.getpass(prompt='Password: ')
    tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
    tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email,
            'client_id': tokenInfo.client_id,
            'client_secret' : tokenInfo.client_secret,
            'password' : tokenInfo.password,
            'grant_type' : 'password',
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s  Run the first cell again and enter the credentials." % (auth_json.get('error_description', auth_json)))
else:
    tokenInfo.access_token = auth_json.get('access_token')
    tokenInfo.refresh_token = auth_json.get('refresh_token')
    if tokenInfo.access_token and tokenInfo.refresh_token:
        print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
    else:
        print("\n\nAuthentication completed but tokens were not returned. Response: {}".format(auth_json))

#print(vars(tokenInfo))
if tokenInfo.access_token and tokenInfo.refresh_token:
    print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)
else:
    print('\n\nNo access token was generated. Check the messages above for errors.')

Enter your XBRL US Web account email: test.tauriello@xbrl.us
Password: ··········
Client ID: ··········
Secret: ··········


Your access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. 

For now, skip ahead to the section 'Make a Query'.


access token: fb435bd1-e674-4dbc-a521-11b31e1967a8 refresh token: e88f5727-385a-456c-af9a-34ea3792f7c4


## Define query filters and fields to return

In [8]:
# Define the parameters for the filter and fields to be returned

endpoint = 'cube'

# Define the parameters of the query

report_section = [" - Statement - "]

# query for a list of 10-K and 10-Q filer entity codes:
# https://api.xbrl.us/api/v1/report/search?report.document-type=10-K,10-K/A,10-Q,10-Q/A&fields=report.entity-name.sort(ASC),entity.code&unique
entity_codes = [
'0000789019'
]

# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
		'period.fiscal-year.sort(DESC)',
		'period.fiscal-period.sort(DESC)',
		'entity.code',
		'report.entity-name',
		'report.id',
		'cube.description.sort(ASC)',
		'cube.tree-sequence.sort(ASC)',
		'cube.primary-local-name',
    'cube.primary-namespace',
		'fact.value',
		'unit',
		'dimensions.count',
		'dimension-pair.sort(ASC)'
        ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 10 # Set as '' to display all rows in the notebook

# Below is the list of what's being queried using the search endpoint.

params = {
         'cube.description': ','.join(report_section),
         'entity.code': ','.join(entity_codes),
         'fields': ','.join(fields)
         }

In [9]:
# @title Loop for all results {"display-mode":"form"}
### Execute the query with loop for all results
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
offset_value = 0
res_df = []
count = 0
query_start = datetime.now()
printed = False
run_query = True

while True:
    if not printed:
        print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query and")
        printed = True
    retry = 0
    while retry < 3:
        res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
        res_json = res.json()
        if 'error' in res_json:
            if res_json['error_description'] == 'Bad or expired token':
                tokenInfo = refresh(tokenInfo)
            else:
                print('There was an error: {}'.format(res_json['error_description']))
                run_query = False
                break
        else:
		        break
        retry +=1
        if retry >= 3:
            print("Can't refresh the access token.  Run the first query block, then rerun the query.")
            run_query = False

    if not run_query:
       break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else:
        offset_value += res_json['paging']['limit']
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"

    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)

    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))

    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html(max_rows=rows_to_display)))

On Wed Oct  1 21:06:00 2025 test.tauriello@xbrl.us (client ID: 69e1257c ...) started the query and
up to 5000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 3815 records.

At Wed Oct  1 21:06:02 2025, the query finished with   3815   rows returned in 0:00:01.982797 for 
https://api.xbrl.us/api/v1/cube/search?unique&cube.description=+-+Statement+-+&entity.code=0000789019&fields=period.fiscal-year.sort(DESC),period.fiscal-period.sort(DESC),entity.code,report.entity-name,report.id,cube.description.sort(ASC),cube.tree-sequence.sort(ASC),cube.primary-local-name,cube.primary-namespace,fact.value,unit,dimensions.count,dimension-pair.sort(ASC)


,period.fiscal-year,period.fiscal-period,entity.code,report.entity-name,report.id,cube.description,cube.tree-sequence,cube.primary-local-name,cube.primary-namespace,fact.value,unit,dimensions.count,dimension-pair
0,2025,Y,0000789019,MICROSOFT CORPORATION,866008,75010 - Statement - INCOME STATEMENTS,1,RevenueFromContractWithCustomerExcludingAssessedTax,http://fasb.org/us-gaap/2024,217778000000,USD,1,[{'ProductOrServiceAxis': 'ServiceOtherMember'}]
1,2025,Y,0000789019,MICROSOFT CORPORATION,866008,75010 - Statement - INCOME STATEMENTS,1,RevenueFromContractWithCustomerExcludingAssessedTax,http://fasb.org/us-gaap/2024,281724000000,USD,0,
2,2025,Y,0000789019,MICROSOFT CORPORATION,866008,75010 - Statement - INCOME STATEMENTS,1,RevenueFromContractWithCustomerExcludingAssessedTax,http://fasb.org/us-gaap/2024,63946000000,USD,1,[{'ProductOrServiceAxis': 'ProductMember'}]
3,2025,Y,0000789019,MICROSOFT CORPORATION,866008,75010 - Statement - INCOME STATEMENTS,2,CostOfGoodsAndServicesSold,http://fasb.org/us-gaap/2024,13501000000,USD,1,[{'ProductOrServiceAxis': 'ProductMember'}]
4,2025,Y,0000789019,MICROSOFT CORPORATION,866008,75010 - Statement - INCOME STATEMENTS,2,CostOfGoodsAndServicesSold,http://fasb.org/us-gaap/2024,74330000000,USD,1,[{'ProductOrServiceAxis': 'ServiceOtherMember'}]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3810,2017,Y,0000789019,MICROSOFT CORPORATION,245526,100060 - Statement - STOCKHOLDERS' EQUITY STATEMENTS,12,StockholdersEquityOther,http://fasb.org/us-gaap/2018-01-31,-86000000,USD,1,[{'StatementEquityComponentsAxis': 'CommonStockIncludingAdditionalPaidInCapitalMember'}]
3811,2017,Y,0000789019,MICROSOFT CORPORATION,245526,100060 - Statement - STOCKHOLDERS' EQUITY STATEMENTS,13,CommonStockDividendsPerShareDeclared,http://fasb.org/us-gaap/2018-01-31,1.56,USD/shares,0,
3812,2016,Y,0000789019,MICROSOFT CORPORATION,245526,100060 - Statement - STOCKHOLDERS' EQUITY STATEMENTS,4,StockholdersEquity,http://fasb.org/us-gaap/2018-01-31,13118000000,USD,1,[{'StatementEquityComponentsAxis': 'RetainedEarningsMember'}]
3813,2016,Y,0000789019,MICROSOFT CORPORATION,245526,100060 - Statement - STOCKHOLDERS' EQUITY STATEMENTS,4,StockholdersEquity,http://fasb.org/us-gaap/2018-01-31,1794000000,USD,1,[{'StatementEquityComponentsAxis': 'AccumulatedOtherComprehensiveIncomeMember'}]


In [7]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
#df.to_csv(r"D:\results.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

from google.colab import drive
drive.mount('drive')
df.to_csv('data.csv')
!cp data.csv "drive/My Drive/"

Mounted at drive
